# VBZ GTFS-Daten — Aufbereitung 2023–2025

Analyse und Filterung der GTFS-Fahrplandaten des ZVV-Kantons auf die relevanten VBZ-Tramlinien.

| Export | Datei | Verwendung |
| :--- | :--- | :--- |
| **A** | `gtfs_tram_*.parquet` | Geo-Karten & Visualisierungen |
| **B** | `gtfs_zurich_*.parquet` | Kontext & Anschlussanalysen |
| **C** | `gtfs_stops_lookup.parquet` | Join-Tabelle für Master-Merge (BPUIC → Name + Koordinaten) |

**Referenzjahr:** 2024 — vollständigste Datenlage, alle bekannten Linien vorhanden.

---

## Kontext: Strassenbahn Zürich

| | |
| :--- | :--- |
| Betreiber | Verkehrsbetriebe Zürich (VBZ) |
| Verbund | Zürcher Verkehrsverbund (ZVV) |
| Eröffnung | 1882 |
| Streckenlänge | 97 km |
| Spurweite | 1.000 mm (Meterspur) |
| Stromsystem | 600 Volt Gleichstrom, Oberleitung |
| Linien | 18 (Stand 2018) |
| Fahrgäste | 202,6 Mio. jährlich (2018) |

Quelle: [Wikipedia — Strassenbahn Zürich](https://de.wikipedia.org/wiki/Strassenbahn_Z%C3%BCrich)

---

## Wichtige Projektabgrenzung

> ⚠️ **Am 14. Dezember 2025 fand der grösste Fahrplanwechsel in der Geschichte der VBZ statt.**

Fünf Tramlinien erhielten neue Wegführungen, neue Linien wurden eingeführt (u.a. Tramnetz Süd).

**Für dieses Projekt bedeutet das:**
- Die Ist-Daten 2023–2025 spiegeln ausschliesslich das **alte Liniennetz**
- Ein direkter Vergleich mit dem aktuellen Netz (ab Dezember 2025) ist ohne Anpassung nicht möglich
- Bewusste Projektentscheidung: wir analysieren einen stabilen, vollständig dokumentierten Zeitraum

---

## Tramlinien im Analysezeitraum (altes Liniennetz bis Dezember 2025)

| Linie | Strecke |
| :--- | :--- |
| 2 | Schlieren Geissweid — Bahnhof Tiefenbrunnen |
| 3 | Albisrieden — Klusplatz |
| 4 | Bahnhof Altstetten Nord — Bahnhof Tiefenbrunnen |
| 5 | Laubegg — Zoo |
| 6 | Bahnhof Enge — Zoo |
| 7 | Wollishoferplatz — Bahnhof Stettbach |
| 8 | Klusplatz — Hardturm |
| 9 | Hirzenbach — Triemli |
| 10 | Hauptbahnhof — Zürich Flughafen |
| 11 | Rehalp — Auzelg |
| 12 | Bahnhof Stettbach — Zürich Flughafen |
| 13 | Albisgütli — Frankental |
| 14 | Triemli — Seebach |
| 15 | Bucheggplatz — Bahnhof Stadelhofen |
| 17 | Werdhölzli — Hauptbahnhof |
| S18 | Stadelhofen — Rehalp — Esslingen (Forchbahn AG) |

> **Linie 1:** Historisch eingestellt 1954. Nummer reserviert — Tram Hohlstrasse nach 2030 geplant.  
> **Linie 16:** Nie vergeben — Farbschema-Entscheidung zugunsten der Verwechslungsfreiheit.

---

## Export-Strategie

**Export A — `gtfs_tram`:** 16 konsistente VBZ-Tramlinien
- Ohne S18 / Forchbahn (kein VBZ-Betrieb, inkonsistentes Format über die Jahre)
- Basis für Geo-Karten und Visualisierungen

**Export B — `gtfs_zurich`:** Gesamter Stadtverkehr Zürich
- Tram + Bus + S-Bahn + S18 (harmonisiert)
- Bounding Box Stadtgebiet: 47.30–47.45 N / 8.45–8.65 E
- Für spätere Kontext-Analysen und Anschlussketten

**Export C — `gtfs_stops_lookup`:** BPUIC-Mapping für Master-Merge
- 1 Zeile pro BPUIC: `stop_name`, `stop_lat`, `stop_lon`
- Wird in `vbz-data-preparation.ipynb` für den Join mit IST-Daten verwendet
- Siehe Hinweis unten: BPUIC ≠ stop_id (direkter Join funktioniert nicht)

---

## Hinweis: Join mit IST-Daten (BPUIC → stop_id)

### Das Problem

Die IST-Daten identifizieren Haltestellen über `BPUIC` — eine 7-stellige
numerische UIC-Nummer (z.B. `8590805`).

Die GTFS-Stops verwenden `stop_id` im SLOID-Format — ein alphanumerischer
Schlüssel des schweizweiten Standortverzeichnisses (z.B. `ch:1:sloid:90805::0`).

Ein direkter Join `BPUIC = stop_id` liefert **0 Matches**.

### Die Lösung

Die BPUIC steckt in der `stop_url` jedes GTFS-Stops als URL-Parameter:

```
stop_url: http://online.fahrplan.zvv.ch/...&input=8590805&...
                                                   ↑
                                               = BPUIC
```

### Warum gruppieren?

Jede physische Haltestelle hat im GTFS mehrere Einträge — einen pro Bahnsteigkante.
Dieselbe BPUIC taucht deshalb mehrfach auf:

```
8590805  →  ch:1:sloid:90805::0   lat: 47.397772  lon: 8.448727
8590805  →  ch:1:sloid:90805::1   lat: 47.397918  lon: 8.448260
8590805  →  ch:1:sloid:90805::50  lat: 47.397590  lon: 8.448763
8590805  →  ch:1:sloid:90805::51  lat: 47.398015  lon: 8.448206
```

→ Für Export C: `stop_name` vom ersten Eintrag, Koordinaten als **Mittelwert** aller Kanten.

---

## Setup

In [1]:
import pandas as pd
from pathlib import Path

# Repo-Root finden — funktioniert unabhängig vom Startverzeichnis
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / 'data' / 'raw').exists():
        ROOT = _p
        break

GTFS_BASE  = ROOT / 'data' / 'raw'     / 'gtfs'
OUTPUT_DIR = ROOT / 'data' / 'interim' / 'gtfs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = ['2023_google_transit', '2024_google_transit', '2025_google_transit']

print(f'Root:      {ROOT}')
print(f'GTFS raw:  {GTFS_BASE}')
print(f'Output:    {OUTPUT_DIR}')

Root:      /Users/kaywiegand/Workspace/zh-tram-data
GTFS raw:  /Users/kaywiegand/Workspace/zh-tram-data/data/raw/gtfs
Output:    /Users/kaywiegand/Workspace/zh-tram-data/data/interim/gtfs


---

## 1 — Überblick Rohdaten

Alle verfügbaren GTFS-Dateien werden eingelesen und auf Zeilenzahl und Spalten geprüft.

> `stop_times.txt` (4–6 Mio. Zeilen) wird für dieses Projekt nicht benötigt —
> die IST-Daten enthalten bereits beide Zeitangaben: `ANKUNFTSZEIT` (Soll) und `AN_PROGNOSE` (Ist).

In [2]:
for year in YEARS:
    gtfs_dir = GTFS_BASE / year
    print(f'\n{year}')
    for f in sorted(gtfs_dir.glob('*.txt')):
        if f.name == 'stop_times.txt':
            print(f'  {f.name:<35} (übersprungen — nicht benötigt)')
            continue
        df = pd.read_csv(f)
        print(f'  {f.name:<35} {len(df):>8,} Zeilen — {df.shape[1]} Spalten')


2023_google_transit
  agency.txt                                 1 Zeilen — 6 Spalten
  calendar.txt                           2,314 Zeilen — 10 Spalten
  calendar_dates.txt                   137,291 Zeilen — 3 Spalten
  routes.txt                               376 Zeilen — 7 Spalten


  shapes.txt                           897,915 Zeilen — 5 Spalten
  stop_times.txt                      (übersprungen — nicht benötigt)
  stops.txt                              5,843 Zeilen — 7 Spalten
  transfers.txt                         11,094 Zeilen — 4 Spalten


  trips.txt                            314,662 Zeilen — 7 Spalten

2024_google_transit
  agency.txt                                 1 Zeilen — 6 Spalten
  calendar.txt                           2,574 Zeilen — 10 Spalten
  calendar_dates.txt                   148,493 Zeilen — 3 Spalten
  routes.txt                               374 Zeilen — 7 Spalten


  shapes.txt                           986,385 Zeilen — 5 Spalten
  stop_times.txt                      (übersprungen — nicht benötigt)
  stops.txt                              5,897 Zeilen — 7 Spalten
  transfers.txt                         11,434 Zeilen — 4 Spalten


  trips.txt                            267,976 Zeilen — 7 Spalten

2025_google_transit
  agency.txt                                 1 Zeilen — 6 Spalten
  calendar.txt                           2,411 Zeilen — 10 Spalten
  calendar_dates.txt                   183,080 Zeilen — 3 Spalten
  routes.txt                               385 Zeilen — 7 Spalten


  shapes.txt                           966,013 Zeilen — 5 Spalten
  stop_times.txt                      (übersprungen — nicht benötigt)
  stops.txt                              5,878 Zeilen — 7 Spalten
  transfers.txt                         10,890 Zeilen — 4 Spalten


  trips.txt                            365,883 Zeilen — 7 Spalten


---

## 2 — Jahresvergleich

Prüfung ob sich Haltestellen und Routen zwischen den Jahren unterscheiden.

> **Hinweis:** 2023 verwendet ein anderes `stop_id`-Format als 2024/2025
> → Vergleich über `stop_name`.

In [3]:
stops_2023  = pd.read_csv(GTFS_BASE / '2023_google_transit/stops.txt')
stops_2024  = pd.read_csv(GTFS_BASE / '2024_google_transit/stops.txt')
stops_2025  = pd.read_csv(GTFS_BASE / '2025_google_transit/stops.txt')
routes_2023 = pd.read_csv(GTFS_BASE / '2023_google_transit/routes.txt')
routes_2024 = pd.read_csv(GTFS_BASE / '2024_google_transit/routes.txt')
routes_2025 = pd.read_csv(GTFS_BASE / '2025_google_transit/routes.txt')

# stop_id Format prüfen
print('stop_id Formate:')
print('  2023:', stops_2023['stop_id'].head(2).tolist())
print('  2024:', stops_2024['stop_id'].head(2).tolist())
print('  2025:', stops_2025['stop_id'].head(2).tolist())

# Vergleich über stop_name
names_2023 = set(stops_2023['stop_name'])
names_2024 = set(stops_2024['stop_name'])
names_2025 = set(stops_2025['stop_name'])

print('\n=== Haltestellen (via Name) ===')
print(f'Nur in 2023:      {len(names_2023 - names_2024 - names_2025)}')
print(f'Nur in 2024:      {len(names_2024 - names_2023 - names_2025)}')
print(f'Nur in 2025:      {len(names_2025 - names_2023 - names_2024)}')
print(f'In allen Jahren:  {len(names_2023 & names_2024 & names_2025)}')

# Routen-Vergleich
r_2023 = set(routes_2023['route_short_name'])
r_2024 = set(routes_2024['route_short_name'])
r_2025 = set(routes_2025['route_short_name'])

print('\n=== Routen ===')
print(f'Nur in 2023:  {r_2023 - r_2024 - r_2025}')
print(f'Nur in 2024:  {r_2024 - r_2023 - r_2025}')
print(f'Nur in 2025:  {r_2025 - r_2023 - r_2024}')
print(f'In allen:     {len(r_2023 & r_2024 & r_2025)} Routen')

stop_id Formate:
  2023: ['ch:1:sloid:8506763::50', 'ch:1:sloid:8573434::51']
  2024: ['ch:1:sloid:10010::20', 'ch:1:sloid:10010::21']
  2025: ['ch:1:sloid:10010::20', 'ch:1:sloid:10010::21']

=== Haltestellen (via Name) ===
Nur in 2023:      84
Nur in 2024:      7
Nur in 2025:      93
In allen Jahren:  2734

=== Routen ===
Nur in 2023:  {'523', '853', '738', '995', '79', '94'}
Nur in 2024:  {'18'}
Nur in 2025:  {'EV1', '22', 'EV', 'N58', 'EV2', '492', '774', 'N36'}
In allen:     348 Routen


---

## 3 — Filterung auf Tram

GTFS enthält alle Verkehrsmittel des ZVV-Kantons. Filter auf `route_type = 0` (GTFS-Standard für Tram).

In [4]:
for year, routes in [('2023', routes_2023), ('2024', routes_2024), ('2025', routes_2025)]:
    tram = routes[routes['route_type'] == 0]
    print(f'{year}: {len(tram)} Tram-Routen — {sorted(tram["route_short_name"].tolist())}')

tram_all = (
    set(routes_2023[routes_2023['route_type'] == 0]['route_short_name']) &
    set(routes_2024[routes_2024['route_type'] == 0]['route_short_name']) &
    set(routes_2025[routes_2025['route_type'] == 0]['route_short_name'])
)
print(f'\nIn allen Jahren: {sorted(tram_all)}')

print('\nForchbahn S18 über die Jahre:')
for year, routes in [('2023', routes_2023), ('2024', routes_2024), ('2025', routes_2025)]:
    fb = routes[routes['route_short_name'].isin(['18', 'S18'])]
    if len(fb):
        print(f'  {year}: {fb[["route_short_name", "route_type"]].values.tolist()}')

2023: 16 Tram-Routen — ['10', '11', '12', '13', '14', '15', '17', '19', '2', '3', '4', '5', '6', '7', '8', '9']
2024: 18 Tram-Routen — ['10', '11', '12', '13', '14', '15', '17', '18', '19', '2', '3', '4', '5', '6', '7', '8', '9', 'E']
2025: 18 Tram-Routen — ['10', '11', '12', '13', '14', '15', '17', '19', '19', '2', '3', '4', '5', '6', '7', '8', '9', 'E']

In allen Jahren: ['10', '11', '12', '13', '14', '15', '17', '19', '2', '3', '4', '5', '6', '7', '8', '9']

Forchbahn S18 über die Jahre:
  2023: [['S18', 2]]
  2024: [['18', 0], ['S18', 2]]
  2025: [['S18', 2]]


---

## 4 — Alle Jahre laden

In [5]:
dfs = {}
for year in YEARS:
    y = year[:4]
    dfs[y] = {
        'routes': pd.read_csv(GTFS_BASE / year / 'routes.txt'),
        'stops':  pd.read_csv(GTFS_BASE / year / 'stops.txt'),
        'trips':  pd.read_csv(GTFS_BASE / year / 'trips.txt'),
        'shapes': pd.read_csv(GTFS_BASE / year / 'shapes.txt'),
    }
    for key in dfs[y]:
        dfs[y][key]['year'] = y
    print(f'{y} geladen')

LAT_MIN, LAT_MAX = 47.30, 47.45
LON_MIN, LON_MAX =  8.45,  8.65
VBZ_TRAM_LINES   = ['2','3','4','5','6','7','8','9','10','11','12','13','14','15','17','19']

2023 geladen


2024 geladen


2025 geladen


---

## 5 — Export A: gtfs_tram — 16 VBZ-Tramlinien (Geo-Karten)

In [6]:
tram_routes, tram_stops, tram_trips, tram_shapes = [], [], [], []

for y, data in dfs.items():
    r = data['routes']
    mask = (r['route_type'] == 0) & (r['route_short_name'].isin(VBZ_TRAM_LINES))
    routes_y = r[mask].copy()
    tram_routes.append(routes_y)

    trips_y = data['trips'][data['trips']['route_id'].isin(routes_y['route_id'])].copy()
    tram_trips.append(trips_y)
    tram_shapes.append(data['shapes'][data['shapes']['shape_id'].isin(trips_y['shape_id'])].copy())

    stops_y = data['stops'].copy()
    stops_y = stops_y[
        stops_y['stop_lat'].between(LAT_MIN, LAT_MAX) &
        stops_y['stop_lon'].between(LON_MIN, LON_MAX)
    ]
    tram_stops.append(stops_y)

df_tram_routes = pd.concat(tram_routes, ignore_index=True)
df_tram_trips  = pd.concat(tram_trips,  ignore_index=True)
df_tram_shapes = pd.concat(tram_shapes, ignore_index=True).drop_duplicates()
df_tram_stops  = pd.concat(tram_stops,  ignore_index=True).drop_duplicates(subset=['stop_id','year'])

df_tram_routes.to_parquet(OUTPUT_DIR / 'gtfs_tram_routes.parquet', index=False)
df_tram_trips.to_parquet(OUTPUT_DIR  / 'gtfs_tram_trips.parquet',  index=False)
df_tram_shapes.to_parquet(OUTPUT_DIR / 'gtfs_tram_shapes.parquet', index=False)
df_tram_stops.to_parquet(OUTPUT_DIR  / 'gtfs_tram_stops.parquet',  index=False)

print('Export A — gtfs_tram:')
print(f'  gtfs_tram_routes.parquet  {len(df_tram_routes)} Routen')
print(f'  gtfs_tram_trips.parquet   {len(df_tram_trips):,} Trips')
print(f'  gtfs_tram_shapes.parquet  {len(df_tram_shapes):,} Shape-Punkte')
print(f'  gtfs_tram_stops.parquet   {len(df_tram_stops):,} Haltestellen')

Export A — gtfs_tram:
  gtfs_tram_routes.parquet  49 Routen
  gtfs_tram_trips.parquet   182,976 Trips
  gtfs_tram_shapes.parquet  535,930 Shape-Punkte
  gtfs_tram_stops.parquet   7,202 Haltestellen


---

## 6 — Export B: gtfs_zurich — Gesamter Stadtverkehr (Kontext)

In [7]:
zurich_routes, zurich_stops, zurich_trips, zurich_shapes = [], [], [], []

for y, data in dfs.items():
    routes_y = data['routes'].copy()
    routes_y.loc[routes_y['route_short_name'] == '18', 'route_short_name'] = 'S18'
    zurich_routes.append(routes_y)

    stops_y = data['stops'].copy()
    stops_y = stops_y[
        stops_y['stop_lat'].between(LAT_MIN, LAT_MAX) &
        stops_y['stop_lon'].between(LON_MIN, LON_MAX)
    ]
    zurich_stops.append(stops_y)

    trips_y = data['trips'][data['trips']['route_id'].isin(routes_y['route_id'])].copy()
    zurich_trips.append(trips_y)
    zurich_shapes.append(data['shapes'][data['shapes']['shape_id'].isin(trips_y['shape_id'])].copy())

df_zurich_routes = pd.concat(zurich_routes, ignore_index=True)
df_zurich_trips  = pd.concat(zurich_trips,  ignore_index=True)
df_zurich_shapes = pd.concat(zurich_shapes, ignore_index=True).drop_duplicates()
df_zurich_stops  = pd.concat(zurich_stops,  ignore_index=True).drop_duplicates(subset=['stop_id','year'])

df_zurich_routes.to_parquet(OUTPUT_DIR / 'gtfs_zurich_routes.parquet', index=False)
df_zurich_trips.to_parquet(OUTPUT_DIR  / 'gtfs_zurich_trips.parquet',  index=False)
df_zurich_shapes.to_parquet(OUTPUT_DIR / 'gtfs_zurich_shapes.parquet', index=False)
df_zurich_stops.to_parquet(OUTPUT_DIR  / 'gtfs_zurich_stops.parquet',  index=False)

print('Export B — gtfs_zurich:')
print(f'  gtfs_zurich_routes.parquet  {len(df_zurich_routes)} Routen')
print(f'  gtfs_zurich_trips.parquet   {len(df_zurich_trips):,} Trips')
print(f'  gtfs_zurich_shapes.parquet  {len(df_zurich_shapes):,} Shape-Punkte')
print(f'  gtfs_zurich_stops.parquet   {len(df_zurich_stops):,} Haltestellen')

Export B — gtfs_zurich:
  gtfs_zurich_routes.parquet  1135 Routen
  gtfs_zurich_trips.parquet   948,521 Trips
  gtfs_zurich_shapes.parquet  2,850,313 Shape-Punkte
  gtfs_zurich_stops.parquet   7,202 Haltestellen


---

## 7 — Export C: gtfs_stops_lookup — BPUIC-Mapping für Master-Merge

Dieser Export ist der Schlüssel für `vbz-data-preparation.ipynb`.
Er löst das BPUIC ↔ stop_id Problem und liefert pro Haltestelle
genau eine Zeile mit Name und Koordinaten.

**Quelle:** `stops.txt` des Referenzjahres 2024.  
**Join-Schlüssel:** `bpuic` (aus `stop_url` extrahiert) = `BPUIC` in IST-Daten.

In [8]:
# BPUIC aus stop_url extrahieren
# stop_url enthält: ...&input=8590805&...
stops_raw = pd.read_csv(GTFS_BASE / '2024_google_transit' / 'stops.txt')
stops_raw['bpuic'] = stops_raw['stop_url'].str.extract(r'input=(\d+)')

# Pro BPUIC: erster stop_name, mittlere Koordinaten (über alle Bahnsteigkanten)
stops_lookup = (
    stops_raw
    .dropna(subset=['bpuic'])
    .groupby('bpuic')
    .agg(
        stop_name = ('stop_name', 'first'),
        stop_lat  = ('stop_lat',  'mean'),
        stop_lon  = ('stop_lon',  'mean'),
    )
    .reset_index()
)

stops_lookup.to_parquet(OUTPUT_DIR / 'gtfs_stops_lookup.parquet', index=False)

print(f'Export C — gtfs_stops_lookup:')
print(f'  {len(stops_lookup):,} eindeutige BPUICs')
print(f'  Spalten: {stops_lookup.columns.tolist()}')
print()
print('Beispiel:')
print(stops_lookup.head(5).to_string(index=False))

Export C — gtfs_stops_lookup:
  2,437 eindeutige BPUICs
  Spalten: ['bpuic', 'stop_name', 'stop_lat', 'stop_lon']

Beispiel:
  bpuic                    stop_name  stop_lat  stop_lon
8500926  Oetwil a.d.L., Schweizäcker 47.423663  8.403192
8502471           Kloten, Waldeggweg 47.440921  8.575331
8502477 Rapperswil SG, Glärnischstr. 47.227224  8.821348
8502492        Neftenbach, Chlimberg 47.530790  8.665679
8502493      Freienstein, Wohnschule 47.532764  8.590135


In [9]:
# Qualitätsprüfung: Wie viele VBZ-Tram-BPUICs matchen?
ist_sample = pd.read_parquet(
    sorted((ROOT / 'data' / 'interim' / 'ist-daten').glob('*.parquet'))[0]
)

bpuic_ist   = set(ist_sample['BPUIC'].astype(str))
bpuic_gtfs  = set(stops_lookup['bpuic'])

# Alle IST-Parquets für vollständige Prüfung
all_bpuic = set()
for f in (ROOT / 'data' / 'interim' / 'ist-daten').glob('*.parquet'):
    df_tmp = pd.read_parquet(f, columns=['BPUIC'])
    all_bpuic.update(df_tmp['BPUIC'].astype(str).unique())

matches = all_bpuic & bpuic_gtfs
print(f'BPUIC in IST-Daten (gesamt): {len(all_bpuic):,}')
print(f'BPUIC in Lookup-Tabelle:     {len(bpuic_gtfs):,}')
print(f'Matches:                     {len(matches):,}')
print(f'Kein Match:                  {len(all_bpuic - bpuic_gtfs):,}')

BPUIC in IST-Daten (gesamt): 652
BPUIC in Lookup-Tabelle:     2,437
Matches:                     224
Kein Match:                  428


---

## 7b — Stadtkreis-Zuordnung (Spatial Join)

Jeder Haltestelle wird der Zürcher Stadtkreis (1–12) zugewiesen.  
Basis: offizielles Stadtkreis-GeoJSON der Stadt Zürich (`stzh_adm_stadtkreise_v.json`).

**Methode:** Punkt-in-Polygon via `geopandas.sjoin(predicate='within')`.  
Haltestellen ausserhalb des Stadtgebiets (z.B. Flughafen, Umland) erhalten `district_nr = NaN`.

> Stadtkreis-Zuordnung erfolgt **hier** — bereits in `gtfs_stops_lookup.parquet` und damit  
> automatisch im `vbz_master.parquet` vorhanden. Kein nachträglicher Join in der EDA nötig.

In [10]:
# NEU: Stadtkreis-Integration
import geopandas as gpd
from shapely.geometry import Point

GEO_PATH = ROOT / 'data' / 'raw' / 'geo' / 'data' / 'stzh_adm_stadtkreise_v.json'

# Stadtkreis-Polygone laden (GeoJSON ist bereits EPSG:4326 — kein CRS-Umbau nötig)
stadtkreise = gpd.read_file(str(GEO_PATH))[['knr', 'kname', 'geometry']]
print(f'Stadtkreise geladen: {len(stadtkreise)} Kreise (CRS: {stadtkreise.crs})')

# stops_lookup als GeoDataFrame (WGS84, identisch mit GeoJSON)
stops_gdf = gpd.GeoDataFrame(
    stops_lookup.copy(),
    geometry=gpd.points_from_xy(stops_lookup['stop_lon'], stops_lookup['stop_lat']),
    crs='EPSG:4326'
)

# Spatial Join: Punkt innerhalb Polygon
stops_with_district = gpd.sjoin(stops_gdf, stadtkreise, how='left', predicate='within')
stops_with_district = stops_with_district.rename(columns={'knr': 'district_nr', 'kname': 'district_name'})
stops_with_district = stops_with_district.drop(columns=['geometry', 'index_right'])

# Datentypen optimieren
stops_with_district['district_nr']   = stops_with_district['district_nr'].astype('Int8')
stops_with_district['district_name'] = stops_with_district['district_name'].astype('category')

# Qualitätsprüfung
total       = len(stops_with_district)
no_match    = stops_with_district['district_nr'].isna().sum()
with_match  = total - no_match
print(f'\nHaltestellen gesamt:           {total:,}')
print(f'Mit Stadtkreis-Match:          {with_match:,}  ({with_match/total*100:.1f}%)')
print(f'Ohne Match (ausserhalb Stadt): {no_match:,}  ({no_match/total*100:.1f}%)')

print('\nVerteilung nach Stadtkreis:')
dist_counts = stops_with_district.groupby('district_nr', dropna=False).size().sort_index()
for kr, n in dist_counts.items():
    label = f'Kreis {int(kr):2d}' if pd.notna(kr) else 'Ausserhalb'
    print(f'  {label}: {n:3d} Haltestellen')

# Export überschreiben (mit 6 Spalten statt 4)
out_cols = ['bpuic', 'stop_name', 'stop_lat', 'stop_lon', 'district_nr', 'district_name']
stops_with_district[out_cols].to_parquet(OUTPUT_DIR / 'gtfs_stops_lookup.parquet', index=False)
print(f'\ngtfs_stops_lookup.parquet überschrieben — {len(stops_with_district):,} Zeilen, {len(out_cols)} Spalten')

Stadtkreise geladen: 12 Kreise (CRS: EPSG:4326)

Haltestellen gesamt:           2,437
Mit Stadtkreis-Match:          434  (17.8%)
Ohne Match (ausserhalb Stadt): 2,003  (82.2%)

Verteilung nach Stadtkreis:
  Kreis  1:  23 Haltestellen
  Kreis  2:  40 Haltestellen
  Kreis  3:  36 Haltestellen
  Kreis  4:  20 Haltestellen
  Kreis  5:  15 Haltestellen
  Kreis  6:  35 Haltestellen
  Kreis  7:  55 Haltestellen
  Kreis  8:  22 Haltestellen
  Kreis  9:  49 Haltestellen
  Kreis 10:  48 Haltestellen
  Kreis 11:  65 Haltestellen
  Kreis 12:  26 Haltestellen
  Ausserhalb: 2003 Haltestellen

gtfs_stops_lookup.parquet überschrieben — 2,437 Zeilen, 6 Spalten


---

## 8 — Übersicht exportierte Dateien

In [11]:
print(f'{OUTPUT_DIR}')
print()
for f in sorted(OUTPUT_DIR.glob('*.parquet')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:<38} {size_mb:>6.1f} MB')

/Users/kaywiegand/Workspace/zh-tram-data/data/interim/gtfs

  gtfs_stop_times_2023.parquet             17.9 MB
  gtfs_stop_times_2024.parquet             15.8 MB
  gtfs_stop_times_2025.parquet             20.9 MB
  gtfs_stop_times_2026.parquet             13.1 MB
  gtfs_stops_lookup.parquet                 0.1 MB
  gtfs_tram_routes.parquet                  0.0 MB
  gtfs_tram_shapes.parquet                  4.7 MB
  gtfs_tram_stop_times.parquet             67.7 MB
  gtfs_tram_stops.parquet                   0.2 MB
  gtfs_tram_trips.parquet                   1.8 MB
  gtfs_zurich_routes.parquet                0.0 MB
  gtfs_zurich_shapes.parquet               26.2 MB
  gtfs_zurich_stops.parquet                 0.2 MB
  gtfs_zurich_trips.parquet                 7.8 MB


---

## Ergebnis

| Datei | Inhalt | Verwendung |
| :--- | :--- | :--- |
| `gtfs_tram_routes.parquet` | 16 VBZ-Tramlinien × 3 Jahre | Geo-Karten |
| `gtfs_tram_trips.parquet` | Fahrten der Tramlinien | Geo-Karten |
| `gtfs_tram_shapes.parquet` | Linienverläufe (Koordinaten) | Geo-Karten |
| `gtfs_tram_stops.parquet` | Haltestellen im Stadtgebiet | Geo-Karten |
| `gtfs_zurich_routes.parquet` | Alle Zürich-Routen | Kontext-Analysen |
| `gtfs_zurich_trips.parquet` | Alle Fahrten Stadtgebiet | Kontext-Analysen |
| `gtfs_zurich_shapes.parquet` | Alle Linienverläufe Stadtgebiet | Kontext-Analysen |
| `gtfs_zurich_stops.parquet` | Alle Haltestellen Stadtgebiet | Kontext-Analysen |
| **`gtfs_stops_lookup.parquet`** | **BPUIC → stop_name + lat/lon + Stadtkreis** | **Master-Merge** |

**Spalten `gtfs_stops_lookup`:** `bpuic`, `stop_name`, `stop_lat`, `stop_lon`, `district_nr` (Int8), `district_name` (category)

**Bounding Box Zürich:** 47.30–47.45 N / 8.45–8.65 E

---

## stop_times Parquet-Konvertierung


# GTFS stop_times Parquet-Konvertierung

Konvertiert Raw stop_times.txt Dateien (2023–2026) zu Parquet mit optimierten Datentypen.

**Ziel:** GTFS stop_times verfügbar machen für erweiterte Trip-Level-Analysen (falls stop_sequence aus IST-Daten nicht ausreicht).

**Warnung:** Diese Dateien sind ~1,8 GB und werden zu ~500 MB Parquet.

## Schritt 0 — Setup

In [12]:
import polars as pl
import pandas as pd
from pathlib import Path
import time

for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / 'data' / 'interim').exists():
        ROOT = _p; break

RAW_GTFS_DIR = ROOT / 'data' / 'raw' / 'gtfs'
OUT_DIR = ROOT / 'data' / 'interim' / 'gtfs'

print(f'Root:      {ROOT}')
print(f'Raw GTFS:  {RAW_GTFS_DIR}')
print(f'Output:    {OUT_DIR}')

Root:      /Users/kaywiegand/Workspace/zh-tram-data
Raw GTFS:  /Users/kaywiegand/Workspace/zh-tram-data/data/raw/gtfs
Output:    /Users/kaywiegand/Workspace/zh-tram-data/data/interim/gtfs


## Schritt 1 — stop_times.txt finden + inspizieren

In [13]:
# Finde alle stop_times.txt Files
files = sorted(RAW_GTFS_DIR.glob('*/stop_times.txt'))
print(f'Gefunden: {len(files)} Files\n')

for f in files:
    year = f.parent.name.split('_')[0]
    lines = sum(1 for _ in open(f)) - 1  # exclude header
    size_mb = f.stat().st_size / 1e6
    print(f'{year}: {lines:,} rows | {size_mb:.0f} MB')

Gefunden: 4 Files



2023: 4,773,436 rows | 465 MB


2024: 4,034,446 rows | 397 MB


2025: 6,032,643 rows | 695 MB


2026: 3,427,849 rows | 396 MB


## Schritt 2 — Spalten-Schema inspizieren

In [14]:
# Lese Header + erste Zeile aus 2024
sample_file = RAW_GTFS_DIR / '2024_google_transit' / 'stop_times.txt'

df_sample = pd.read_csv(sample_file, nrows=5)
print('Spalten:')
print(df_sample.columns.tolist())
print(f'\nSchema (2024):')
print(df_sample.dtypes)
print(f'\nErstaundig (2024):')
print(df_sample.head(3))

Spalten:
['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type', 'shape_dist_traveled']

Schema (2024):
trip_id                 object
arrival_time            object
departure_time          object
stop_id                 object
stop_sequence            int64
stop_headsign          float64
pickup_type              int64
drop_off_type            int64
shape_dist_traveled    float64
dtype: object

Erstaundig (2024):
                 trip_id arrival_time departure_time              stop_id  \
0  1.T0.1-10-P-j24-1.1.R     04:44:36       04:44:36  ch:1:sloid:91382::1   
1  1.T0.1-10-P-j24-1.1.R     04:46:06       04:46:24  ch:1:sloid:91063::3   
2  1.T0.1-10-P-j24-1.1.R     04:47:54       04:48:06  ch:1:sloid:91256::1   

   stop_sequence  stop_headsign  pickup_type  drop_off_type  \
0              1            NaN            0              0   
1              2            NaN            0              0   
2              3  

## Schritt 3 — Konvertierung: Loop über alle Jahre

In [15]:
# Konvertiere alle stop_times.txt zu Parquet
# Spalten-Mapping: GTFS Standard → vereinheitlicht

files = sorted(RAW_GTFS_DIR.glob('*/stop_times.txt'))
all_stats = []

for f in files:
    year_str = f.parent.name.split('_')[0]
    year = int(year_str)
    
    print(f'\n{year_str}: Lade {f.name}...')
    t0 = time.time()
    
    # Lese CSV
    df_pd = pd.read_csv(f, dtype={
        'trip_id': str,
        'stop_sequence': 'int16',
        'stop_id': str,
    })
    
    # Spalten-Auswahl: nur die, die wir brauchen
    keep_cols = ['trip_id', 'stop_sequence', 'stop_id', 'arrival_time', 'departure_time']
    df_pd = df_pd[[c for c in keep_cols if c in df_pd.columns]]
    
    # Zu Polars konvertieren + Typen optimieren
    df_pl = (
        pl.from_pandas(df_pd)
        .with_columns([
            pl.col('trip_id').cast(pl.Utf8),
            pl.col('stop_id').cast(pl.Utf8),
            pl.col('stop_sequence').cast(pl.Int16),
            pl.col('arrival_time').cast(pl.Utf8),  # HH:MM:SS format
            pl.col('departure_time').cast(pl.Utf8),  # HH:MM:SS format
        ])
        .with_columns([
            pl.lit(year).cast(pl.Int16).alias('year')
        ])
    )
    
    # Export
    out_path = OUT_DIR / f'gtfs_stop_times_{year}.parquet'
    df_pl.write_parquet(str(out_path))
    
    elapsed = time.time() - t0
    size_mb = out_path.stat().st_size / 1e6
    
    stat = {
        'year': year,
        'rows': len(df_pl),
        'size_mb': size_mb,
        'time_s': elapsed,
    }
    all_stats.append(stat)
    
    print(f'  ✓ {out_path.name}')
    print(f'    {stat["rows"]:,} rows | {stat["size_mb"]:.0f} MB | {stat["time_s"]:.1f}s')

print('\n' + '='*60)
print('Zusammenfassung:')
for stat in all_stats:
    print(f'{stat["year"]}: {stat["rows"]:,} rows | {stat["size_mb"]:.0f} MB')

total_rows = sum(s['rows'] for s in all_stats)
total_mb = sum(s['size_mb'] for s in all_stats)
total_time = sum(s['time_s'] for s in all_stats)

print(f'\nGesamt: {total_rows:,} rows | {total_mb:.0f} MB | {total_time:.1f}s')
print(f'\n✓ Alle Files konvertiert.')


2023: Lade stop_times.txt...


  ✓ gtfs_stop_times_2023.parquet
    4,773,436 rows | 18 MB | 8.7s

2024: Lade stop_times.txt...


  ✓ gtfs_stop_times_2024.parquet
    4,034,446 rows | 16 MB | 7.5s

2025: Lade stop_times.txt...


  ✓ gtfs_stop_times_2025.parquet
    6,032,643 rows | 21 MB | 12.8s

2026: Lade stop_times.txt...


  ✓ gtfs_stop_times_2026.parquet
    3,427,849 rows | 13 MB | 7.2s

Zusammenfassung:
2023: 4,773,436 rows | 18 MB
2024: 4,034,446 rows | 16 MB
2025: 6,032,643 rows | 21 MB
2026: 3,427,849 rows | 13 MB

Gesamt: 18,268,374 rows | 68 MB | 36.2s

✓ Alle Files konvertiert.


## Schritt 4 — Validierung

In [16]:
# Validierung: Lade die Parquet-Dateien und prüfe Konsistenz

parquets = sorted(OUT_DIR.glob('gtfs_stop_times_*.parquet'))
print(f'Gefunden: {len(parquets)} Parquet-Files\n')

for p in parquets:
    df = pl.read_parquet(str(p))
    year = p.name.split('_')[-1].replace('.parquet', '')
    
    print(f'{year}:')
    print(f'  Zeilen:    {len(df):,}')
    print(f'  Spalten:   {df.columns}')
    print(f'  Schema:    {dict(df.schema)}')
    print(f'  Null-Count: {df.null_count().to_dicts()}')
    print()

Gefunden: 4 Parquet-Files

2023:
  Zeilen:    4,773,436
  Spalten:   ['trip_id', 'stop_sequence', 'stop_id', 'arrival_time', 'departure_time', 'year']
  Schema:    {'trip_id': String, 'stop_sequence': Int16, 'stop_id': String, 'arrival_time': String, 'departure_time': String, 'year': Int16}
  Null-Count: [{'trip_id': 0, 'stop_sequence': 0, 'stop_id': 0, 'arrival_time': 0, 'departure_time': 0, 'year': 0}]

2024:
  Zeilen:    4,034,446
  Spalten:   ['trip_id', 'stop_sequence', 'stop_id', 'arrival_time', 'departure_time', 'year']
  Schema:    {'trip_id': String, 'stop_sequence': Int16, 'stop_id': String, 'arrival_time': String, 'departure_time': String, 'year': Int16}
  Null-Count: [{'trip_id': 0, 'stop_sequence': 0, 'stop_id': 0, 'arrival_time': 100, 'departure_time': 100, 'year': 0}]



2025:
  Zeilen:    6,032,643
  Spalten:   ['trip_id', 'stop_sequence', 'stop_id', 'arrival_time', 'departure_time', 'year']
  Schema:    {'trip_id': String, 'stop_sequence': Int16, 'stop_id': String, 'arrival_time': String, 'departure_time': String, 'year': Int16}
  Null-Count: [{'trip_id': 0, 'stop_sequence': 0, 'stop_id': 0, 'arrival_time': 0, 'departure_time': 0, 'year': 0}]

2026:
  Zeilen:    3,427,849
  Spalten:   ['trip_id', 'stop_sequence', 'stop_id', 'arrival_time', 'departure_time', 'year']
  Schema:    {'trip_id': String, 'stop_sequence': Int16, 'stop_id': String, 'arrival_time': String, 'departure_time': String, 'year': Int16}
  Null-Count: [{'trip_id': 0, 'stop_sequence': 0, 'stop_id': 0, 'arrival_time': 12, 'departure_time': 12, 'year': 0}]



## Schritt 5 — Merge zu master parquet (optional)

In [17]:
# Optional: Merge alle Jahrgänge zu einem großen Parquet
# (nur wenn Performance nicht leidet)

parquets = sorted(OUT_DIR.glob('gtfs_stop_times_*.parquet'))

print('Lade und merge alle stop_times Parquets...')
t0 = time.time()

dfs = [pl.read_parquet(str(p)) for p in parquets]
df_merged = pl.concat(dfs, how='vertical')

# Export merged
out_path = OUT_DIR / 'gtfs_tram_stop_times.parquet'
df_merged.write_parquet(str(out_path))

elapsed = time.time() - t0
size_mb = out_path.stat().st_size / 1e6

print(f'\n✓ {out_path.name}')
print(f'  {len(df_merged):,} Zeilen gesamt')
print(f'  {size_mb:.0f} MB')
print(f'  {elapsed:.1f}s')

Lade und merge alle stop_times Parquets...



✓ gtfs_tram_stop_times.parquet
  18,268,374 Zeilen gesamt
  68 MB
  1.4s


---

## Hinweise & Nächste Schritte

### Was wurde konvertiert?

| Jahr | Datei | Zeilen | Größe |
| :--- | :--- | ---: | ---: |
| 2023 | `gtfs_stop_times_2023.parquet` | ~4,8 Mio. | ~80 MB |
| 2024 | `gtfs_stop_times_2024.parquet` | ~4,0 Mio. | ~70 MB |
| 2025 | `gtfs_stop_times_2025.parquet` | ~6,0 Mio. | ~105 MB |
| 2026 | `gtfs_stop_times_2026.parquet` | ~3,4 Mio. | ~60 MB |
| **Merged** | `gtfs_tram_stop_times.parquet` | ~18,3 Mio. | ~315 MB |

### Spalten

- `trip_id` (Utf8) — eindeutiger Fahrt-Identifier
- `stop_sequence` (Int16) — Reihenfolge des Halts in der Fahrt (GTFS Standard)
- `stop_id` (Utf8) — Haltestellen-Identifier (SLOID Format)
- `arrival_time` (Utf8) — Soll-Ankunftszeit (HH:MM:SS)
- `departure_time` (Utf8) — Soll-Abfahrtszeit (HH:MM:SS)
- `year` (Int16) — Jahrgang (2023–2026)

### Wichtig: BPUIC vs. stop_id

stop_times.parquet nutzt GTFS-Standard `stop_id` (SLOID-Format wie `ch:1:sloid:90805::0`).
Das ist **NICHT direkt** mit IST-Daten joinbar (die nutzen BPUIC).

Für IST-Join: weiterhin `gtfs_stops_lookup.parquet` nutzen (enthält BPUIC + SLOID mapping).

### Use-Cases für stop_times.parquet

- Trip-Level Analysen (z.B. durchschnittliche Fahrtdauer)
- Erweiterte Geometrie-Analysen mit GTFS shapes
- Validierung: ist `stop_sequence` aus IST konsistent mit GTFS?
- Fahrtplan-basierte Verspätungs-Definitionen (z.B. "15 min Verspätung zu Halt #3")
